In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

## ELC data

In [3]:
df_input = pd.read_excel('./data/ELC_dataset.xlsx',sheet_name='Data')

In [4]:
# drop columns
df_ELC = df_input.drop(columns=["LEVEL","ELC_UNIT", "ELC_SOURCE"]).copy()

# keep data only from 2021 to 2023
df_ELC = df_ELC[(df_input["YEAR"].between(2021, 2023))]

# keep specific copuntries
eu = [
    "AT","BE","HR","CY","CZ","DK","EE","FI","FR","DE",
    "HU","IE","IT","LV","LT","LU","MT","NL","PL","PT","SK",
    "SI","ES","SE"
]
extra = ["NO", "CH", "UK"]
countries = eu + extra

df_ELC = df_ELC[
    df_ELC["NUTS"].str[:2].isin(countries)
]

In [5]:
# aggregate nuts 3 to nuts 2
df_ELC["NUTS"] = df_ELC["NUTS"].apply(
    lambda x: x[:-1] if isinstance(x, str) and len(x) == 5 else x
)

df_ELC = (
    df_ELC
    .groupby(["NUTS", "YEAR"], as_index=False)["ELC"]
    .sum()
)

# drop oveseas territories 
# FRY1, FRY2, FRY3, FRY4, FRY5, PT20, PT30, ES70
df_ELC = df_ELC[
    ~df_ELC["NUTS"].str.startswith(("FRY", "PT20", "PT30", "ES70"))
]


In [6]:
# adjuts nuts2 level for country where nuts0 or nuts1 is equal to nuts2
df_ELC["LEVEL"] = df_ELC["NUTS"].str.len().map({2: 0, 3: 1, 4: 2})

mapping = {
    "CY": "CY00",
    "EE": "EE00",
    "HR": "HR0",
    "LT": "LT0",
    "LU": "LU00",
    "LV": "LV00",
    "MT": "MT00",
    "DE3": "DE30",
    "DE4": "DE40",
    "DE5": "DE50",
    "DE6": "DE60",
    "DE8": "DE80",
    "DEC": "DEC0",
    "DEE": "DEE0",
    "DEF": "DEF0",
    "DEG": "DEG0", 
}

df_ELC["NUTS"] = df_ELC["NUTS"].replace(mapping)
df_ELC["LEVEL"] = df_ELC["NUTS"].str.len().map({2: 0, 3: 1, 4: 2})
print(df_ELC["LEVEL"].value_counts()/3)



LEVEL
2    222.0
1     10.0
Name: count, dtype: float64


## NUTS geometry data

In [8]:
geo_df_input = gpd.read_file('./data/NUTS_RG_20M_2021_3035.gpkg')

In [9]:
# merge
merged = df_ELC.merge(
    geo_df_input[['NUTS_ID', 'geometry']].rename(columns={'NUTS_ID': 'NUTS'}),
    on='NUTS',
    how='left'
)

merged = gpd.GeoDataFrame(merged, geometry='geometry', crs=geo_df_input.crs)

# area in km2
merged['AREA'] = merged.geometry.area / 1_000_000

## Population data

In [12]:
df_population_input = pd.read_csv('./data/population.csv')

In [13]:
# melt
df_population = df_population_input.melt(
    id_vars='NUTS',
    var_name='YEAR',
    value_name='POPULATION'
)

# force YEAR to int
df_population["YEAR"] = df_population["YEAR"].astype(int)

# merge
merged = pd.merge(merged, df_population, on=['NUTS','YEAR'], how='left')

## GDP data

In [15]:
df_gdp_input = pd.read_csv('./data/GDP.csv')

In [16]:
# melt
df_gdp = df_gdp_input.melt(
    id_vars='NUTS',
    var_name='YEAR',
    value_name='GDP'
)

# force YEAR to int
df_gdp["YEAR"] = df_gdp["YEAR"].astype(int)

# merge
merged = pd.merge(merged, df_gdp, on=['NUTS','YEAR'], how='left')

## Real compensation data

In [18]:
df_compensation_input = pd.read_csv('./data/real_compensation.csv')

In [23]:
# melt
df_compensation = df_compensation_input.melt(
    id_vars='NUTS',
    var_name='YEAR',
    value_name='COMPENSATION'
)

# force YEAR to int
df_compensation["YEAR"] = df_compensation["YEAR"].astype(int)

# merge 
merged = pd.merge(merged, df_compensation, on=['NUTS','YEAR'], how='left')

## CDD data

In [25]:
df_cdd_input = pd.read_csv('./data/CDD.csv')

In [26]:
# melt
df_cdd = df_cdd_input.melt(
    id_vars='NUTS',
    var_name='YEAR',
    value_name='CDD'
)

# force YEAR to int
df_cdd["YEAR"] = df_cdd["YEAR"].astype(int)

# merge 
merged = pd.merge(merged, df_cdd, on=['NUTS','YEAR'], how='left')

## HDD data

In [28]:
df_hdd_input = pd.read_csv('./data/HDD.csv')

In [32]:
# melt
df_hdd = df_hdd_input.melt(
    id_vars='NUTS',
    var_name='YEAR',
    value_name='HDD'
)

# force YEAR to int
df_hdd["YEAR"] = df_hdd["YEAR"].astype(int)

# merge 
merged = pd.merge(merged, df_hdd, on=['NUTS','YEAR'], how='left')

## Additional CDD and HDD data for CH and UK

In [39]:
df_dd_input = pd.read_csv('./data/dd.csv', sep=';')

In [43]:
# keep only period 2021-2023
df = df_dd_input[df_dd_input["YEAR"].between(2021, 2023)]

# keep NUTS 2
df = df[df["NUTS_CODE"].str.len() == 4]

# aggregate monthly -> yearly
df_annual = (
    df.groupby(['NUTS_CODE', 'YEAR'], as_index=False)[['CDD', 'HDD']]
      .sum()
)

# rename columns before merging
df_annual = df_annual.rename(columns={'NUTS_CODE': 'NUTS', 'CDD': 'CDD_new', 'HDD': 'HDD_new'})

# merge 
merged = merged.merge(
    df_annual,
    on=['NUTS', 'YEAR'],
    how='left',
)

# fill 
merged['CDD'] = merged['CDD'].fillna(merged['CDD_new'])
merged['HDD'] = merged['HDD'].fillna(merged['HDD_new'])

merged = merged.drop(columns=['CDD_new', 'HDD_new'])

# Output

In [48]:
# Create variables
merged['ELC_cap'] = merged['ELC'] / merged['POPULATION'] * 1_000 # MWh per person
merged['GDP'] = merged['GDP'] / 1_000 # th euro per person
merged['COMPENSATION'] = merged['COMPENSATION'] / 1_000 # th euro per employee 
merged['POP_density'] = merged['POPULATION'] / merged['AREA'] # people per KM2

# rename variables
merged = merged.rename(columns={'GDP': 'GDP_cap', 'COMPENSATION': 'COMP_emp'})

In [50]:
# check na
print(merged.isna().sum().sum())

# reset index
merged = merged.reset_index(drop=True)

# save CSV
merged.to_csv("data/dataset.csv", index=False)


0
